# COMPASS multivariate models

Elastic-net Cox and XGBoost survival:cox, each in "both" (full feature set)
and "baseline" (age-only) configurations, for time to platinum on the full ADT
cohort at the 0/+90/+180-day landmarks. This matches the Dynamic-DeepHit
setup in `03b_multivariate_longitudinal.ipynb`.
Requires `01_preprocessing.ipynb` to have built the merged `profile_data`
inputs under `prediction_inputs_<arm>/` first. The available-case sensitivity
compares labs with Gleason on the matched Gleason-versus-labs cohort at each
treatment landmark; the clinical combinations fit every labs/Gleason/somatic
combination on the matched Gleason + somatic cohort.
Predictive checklists for platinum and NEPC run last, through their own CLI.

In [ ]:
ARMS = ["adt"]
# Time-to-platinum only, matching 03b_multivariate_longitudinal.ipynb.
ENDPOINTS = ("platinum",)          # was: ("platinum", "nepc")
# Multivariable prediction runs on the full ADT cohort only. The cohort /
# exclusion sensitivity cross is reported for the univariate associations
# (02_univariate.ipynb); restore the commented values to widen it here too.
COHORTS = ("all",)                 # full arm cohort, no MRN restriction
# COHORTS = ("all", "metastatic_adt", "metastatic_llm")
#   metastatic_adt -- retrospective ADT-intent stratum
#   metastatic_llm -- met_diagnosis LLM metastatic status
# Orthogonal to COHORTS: "none" keeps every patient, "pre_adt_castrate"
# drops those with a castrate testosterone (<50 ng/dL) before ADT start.
EXCLUSIONS = ("none",)             # was: ("none", "pre_adt_castrate")
OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

import sys
sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.FORCE_RERUN = OVERWRITE
RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS)
print(f"{len(RUNS)} run(s): {[r['label'] for r in RUNS]}")
print(f"landmarks: {sorted({lm for r in RUNS for lm in r['landmarks']})}")
print(f"models: {sorted({(m, c) for m, c, _ in cp.MULTIVARIATE_TASK_SPECS})}")

## Run multivariate models

Elastic-net (both/baseline) and XGBoost (both/baseline) arms. Set
`OVERWRITE = True` in the configuration cell to refit and replace existing outputs. With `False`, completed tasks are skipped.

In [ ]:
for run in RUNS:
    cp.run_multivariate(run)

## Available-case Gleason sensitivity

Fit one matched comparison on the Gleason cohort: labs, Gleason alone, and
Gleason + labs. Somatic calls are compared with labs only in the clinical
combinations below. The cohort preserves the landmark-specific split and ADT-relative follow-up. Gleason uses
the score closest to each landmark among values documented by then. It runs at 0, +90,
and +180 days; it is not comparable to the observation-indexed univariate analyses.

In [ ]:
for run in RUNS:
    cp.run_multivariate_available_case_sensitivity(run)

### Available-case sensitivity results

Held-out AUC(t), C-index, and Brier metrics for the matched Gleason cohort.

In [ ]:
import pandas as pd

sensitivity_rows = []
for run in RUNS:
    for analysis, feature_sets in (
        ('gleason', ('labs', 'gleason', 'gleason-labs')),
    ):
        for landmark_day in run['landmarks']:
            for feature_set, (model, filename) in (
                (feature_set, model_file)
                for model_file in (('cox', 'cox_agg_multivariable_metrics.csv'), ('xgboost', 'landmark_xgboost_metrics.csv'))
                for feature_set in feature_sets
            ):
                path = run['output_dir'] / 'sensitivity_available_case' / analysis / feature_set / model / f'landmark_{landmark_day}' / 'both' / filename
                if path.exists():
                    frame = pd.read_csv(path)
                    frame['analysis'] = analysis
                    frame['feature_set'] = feature_set
                    frame['source_path'] = str(path)
                    sensitivity_rows.append(frame)
available_case_sensitivity_df = pd.concat(sensitivity_rows, ignore_index=True) if sensitivity_rows else pd.DataFrame()
available_case_sensitivity_df

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
config), then combined across runs.

In [ ]:
summary_dfs = {cp.run_key(run): cp.summarize_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
# Dynamic DeepHit (03b, torch-gated) shares the canonical metrics schema, so its
# headline rows concatenate directly with the Cox/XGBoost tables above. Runs
# without 03b outputs simply report status "missing" for these rows.
for run in RUNS:
    deephit = cp.summarize_longitudinal_outputs(run)
    deephit = deephit[deephit["model"] == "dynamic-deephit-dyn"]
    if deephit.empty:
        continue
    key = cp.run_key(run)
    summary_dfs[key] = pd.concat([summary_dfs[key], deephit], ignore_index=True)
    print(f"=== {key}: with dynamic DeepHit ===")
    display(summary_dfs[key])


In [ ]:
import pandas as pd

combined_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_summary_df

## Held-out per-patient risk scores

Every multivariable fit writes one row per held-out patient next to its metrics
(`*_patient_risks.csv`): the risk score, the duration, and the event indicator.
The file can carry two scoring schemes, distinguished by its `dataset` column,
and they are **not interchangeable**:

| `dataset` | Covers | Scored by |
|---|---|---|
| `test` | the held-out test block | the final model, refit on all of train/val |
| `cv_oof` | **every patient in the cohort** | the one outer CV fold that excluded them |

`test` is always written. `cv_oof` appears only when the fit was run with
`--out-of-fold-risks`, which is what gives a held-out score to every patient
rather than the test block alone. It re-tunes hyperparameters inside each outer
fold, so it costs roughly `--oof-outer-folds` times a normal run; reusing the
ordinary tuning folds instead would be far cheaper but leaky, since those same
folds choose the winning penalizer and would then be scoring the patients that
chose it.

These feed the risk-stratified figures (`risk_score_stratified_figures.py`),
which select one scheme via `--dataset` and compare the score against Gleason,
stage, and the TP53/PTEN/RB1 trio. Render the two to separate output
directories: they come from different models, so pooling them into one KM panel
would mix two score scales.

The cell below reports coverage rather than refitting. `n_patients` is the test
block; `n_oof_patients`/`n_oof_scored` are the out-of-fold coverage and read 0
for runs without `--out-of-fold-risks`. **A task whose metrics exist but whose
risk file is `missing` was fit before the risk-score output existed**: the
resume logic keys on the metrics file, so it will keep being skipped. Set
`OVERWRITE = True` in the configuration cell and re-run the fitting cell above
to regenerate those.

In [ ]:
risk_dfs = {cp.run_key(run): cp.summarize_patient_risks(run) for run in RUNS}
for label, df in risk_dfs.items():
    print(f"=== {label} ===")
    display(df.drop(columns=["path"]))

combined_risk_df = pd.concat(risk_dfs.values(), ignore_index=True)
missing = combined_risk_df.loc[combined_risk_df["status"] == "missing"]
if missing.empty:
    n_ok = int((combined_risk_df["status"] == "ok").sum())
    print(f"All {n_ok} multivariable tasks wrote held-out risk scores.")
else:
    print(
        f"{len(missing)} of {len(combined_risk_df)} tasks have no risk file. "
        "These were fit before the risk-score output existed; set OVERWRITE = True "
        "and re-run the fitting cell to regenerate them:"
    )
    display(missing[["run", "model", "landmark", "config", "endpoint"]])

# Out-of-fold coverage, reported apart from the test block because the two come
# from different models. A task with n_oof_patients == 0 simply was not run with
# --out-of-fold-risks; only a shortfall BETWEEN n_oof_scored and n_oof_patients
# means an outer fold failed to fit and some patients went unscored.
scored = combined_risk_df.loc[combined_risk_df["status"] == "ok"]
with_oof = scored.loc[scored["n_oof_patients"] > 0]
if with_oof.empty:
    print(
        "\nNo task carries full-cohort out-of-fold scores. Re-run the fitting "
        "cell with --out-of-fold-risks to give every patient a held-out score "
        "instead of the test block alone."
    )
else:
    short = with_oof.loc[with_oof["n_oof_scored"] < with_oof["n_oof_patients"]]
    print(
        f"\n{len(with_oof)} of {len(scored)} completed tasks carry full-cohort "
        f"out-of-fold scores."
    )
    if not short.empty:
        print(
            f"{len(short)} of those have unscored patients -- an outer fold "
            f"failed to fit, so the figures would be drawn on a subset:"
        )
        display(
            short[["run", "model", "landmark", "config",
                   "n_oof_scored", "n_oof_patients"]]
        )

## Gleason / somatic / labs component combinations (Plan §2c)

Every non-empty combination of {labs, gleason, somatic} (all 7) on the matched
`gleason_somatic_available_case` cohort, both model families. The Gleason-only
cohort's arms are not refit here: they are the available-case sensitivity above.
Requires `build_somatic_gleason_inputs.py`'s `main()` to have written the
`somatic_gleason/<cohort>_available_case/` inputs first. Resumes per-arm like
the sensitivity cell above.

In [ ]:
for run in RUNS:
    cp.run_multivariate_clinical_combinations(run)

### Clinical combination results

One row per (cohort, arm, model, landmark): held-out metrics plus
`delta_c_index` (and its bootstrap CI) versus that cohort's `labs` arm,
computed on the shared test-block patients (exact pairing, since every arm in
a cohort is fit on the same available-case patients).

In [ ]:
clinical_combination_dfs = {cp.run_key(run): cp.summarize_clinical_combinations(run) for run in RUNS}
for label, df in clinical_combination_dfs.items():
    print(f"=== {label} ===")
    display(df)

combined_clinical_combinations_df = pd.concat(clinical_combination_dfs.values(), ignore_index=True)
combined_clinical_combinations_df

## Risk-score stratification by held-out clinical/genomic features (Plan §4)

Stratifies the labs risk score by Gleason, stage, and the TP53/PTEN/RB1 trio
(`build_somatic_gleason_inputs.py`'s `clinical_stratifiers_landmark{D}.csv`,
Plan §4a) on the full cohort, and separately compares the labs arm against
every matched clinical/somatic arm within the
`gleason_somatic_available_case` cohort (using `--comparison-risks`, Plan
§4b). Runs once per scoring scheme (`test`, `cv_oof`) into separate
subdirectories under `risk_stratification/<scheme>/...` -- **the two schemes
are never pooled**. `cv_oof` rows are skipped with a warning for any run not
fit with `--out-of-fold-risks`. Requires `run_multivariate` and
`run_multivariate_clinical_combinations` above to have completed first.

In [ ]:
for run in RUNS:
    cp.run_risk_stratification(run)

## Predictive checklists

Short M-of-N checklists, one per endpoint, landmark, and horizon, each with matched
age-only Cox, elastic-net Cox, and XGBoost comparators fit on the same patients. Both
configs run through the `predictive_checklists` CLI as subprocesses, so no model is fit
in a notebook cell. See `predictive_checklists/README.md` for the censoring estimand,
calibration, and solver details. Endpoints come from each config (platinum and NEPC),
not from `ENDPOINTS` above.

| Run | Config | Cohort | Clock | Landmarks | Horizons |
|---|---|---|---|---|---|
| `full_adt` | `default_config.json` | full ADT cohort | ADT start | 0 (primary), +90, +180 days | 1-5 years |
| `gleason_trio` | `gleason_trio_config.json` | `gleason_somatic_available_case` | sequencing report date (`REPORT_DT`) | sequencing + 0/90/180 days | 5 years |

`full_adt` adds a resolved-label NEPC sensitivity analysis.

`gleason_trio` starts from the report date of the specimen closest to ADT start. Labs
and Gleason must be available by each landmark, the base split is inherited, and age is
age at sequencing. Candidates add `GLEASON_SCORE` and `TP53_ANY` / `RB1_ANY` / `PTEN_ANY`
(any SNV/DEL/AMP/SV on the anchoring specimen) to the labs and age, and the comparators
get the same predictors. `min_events` is 5, not 10, because this subset has few NEPC
events, and the resolved-label NEPC sensitivity is off.

In [ ]:
from pathlib import Path
import json
import subprocess

RUN_CHECKLIST_PREFLIGHT = True
RUN_CHECKLISTS = False  # set True after reviewing the preflight counts below
CHECKLIST_OVERWRITE = False  # True: archive previous results and refit; False: resume
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "survival_common").is_dir())
CHECKLIST_SUBPROJECT = REPO_ROOT / "COMPASS/multivariable_models/predictive_checklists"
INPUTS_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
CHECKLIST_ROOT = INPUTS_ROOT / "predictive_checklists"
GLEASON_TRIO_SUBSET = "somatic_gleason/gleason_somatic_available_case"

CHECKLIST_RUNS = {
    # name: (config, input args, landmarks, horizons, preflight output, run output)
    "full_adt": (
        CHECKLIST_SUBPROJECT / "default_config.json",
        ["--inputs-root", str(INPUTS_ROOT)],
        [0, 90, 180], [365, 730, 1095, 1460, 1825],
        CHECKLIST_ROOT / "preflight", CHECKLIST_ROOT / "full_adt",
    ),
    "gleason_trio": (
        CHECKLIST_SUBPROJECT / "gleason_trio_config.json",
        ["--platinum-inputs", str(INPUTS_ROOT / "prediction_inputs_adt" / GLEASON_TRIO_SUBSET),
         "--nepc-inputs", str(INPUTS_ROOT / "prediction_inputs_adt_nepc" / GLEASON_TRIO_SUBSET)],
        [0, 90, 180], [1825],
        CHECKLIST_ROOT / "preflight_gleason_trio", CHECKLIST_ROOT / "gleason_trio_labs",
    ),
}


def checklist_command(config, input_args, landmarks, horizons, output):
    return [sys.executable, "-m", "COMPASS.multivariable_models.predictive_checklists",
            "--config", str(config), *input_args,
            "--landmarks", *map(str, landmarks), "--horizons", *map(str, horizons),
            "--output-dir", str(output)]

### Validate cohort inputs

Requires the endpoint manifests from `01_preprocessing.ipynb`. Preflight checks the
scope, endpoint identity, and shared patient splits of every requested input, then
reports the known cases and controls at each horizon. It fits nothing, so it is rebuilt
on every run. Review the NEPC counts before training: an underpowered horizon is
recorded as such, not fitted.

In [ ]:
if RUN_CHECKLIST_PREFLIGHT:
    for name, (config, input_args, landmarks, horizons, preflight, _) in CHECKLIST_RUNS.items():
        print(f"[run ] checklist preflight {name} -> {preflight}")
        # Always --overwrite: the CLI refuses an existing output directory otherwise.
        subprocess.run(checklist_command(config, input_args, landmarks, horizons, preflight)
                       + ["--preflight-only", "--overwrite"], cwd=REPO_ROOT, check=True)

In [ ]:
preflight_rows = []
for name, (*_, preflight, _) in CHECKLIST_RUNS.items():
    preflight_path = preflight / "preflight.json"
    if not preflight_path.exists():
        continue
    for cell in json.loads(preflight_path.read_text()):
        for horizon in cell.get("requested_horizons", [{}]):
            preflight_rows.append({"checklist_run": name, "endpoint": cell["endpoint"],
                                   "landmark_days": cell["landmark_days"],
                                   "n_patients": cell.get("n_patients"),
                                   "events_by_split": cell.get("events_by_split"),
                                   "horizon_days": horizon.get("horizon_days"),
                                   "status": horizon.get("status", cell["status"]),
                                   "reason": horizon.get("reason", cell.get("reason"))})
pd.DataFrame(preflight_rows)

### Train and evaluate

Set `RUN_CHECKLISTS = True` after reviewing preflight. The test block stays held out.
Runtime scales with checklist lengths, folds, endpoints, landmarks, and horizons, so
begin with landmark 0 and one horizon for a solver pilot. With
`CHECKLIST_OVERWRITE = False`, a run reuses its completed horizons and starts fresh if
none exist. The CLI refuses to resume if the config, inputs, or checklist source have
changed; set `CHECKLIST_OVERWRITE = True` to archive that run and refit.

In [ ]:
if RUN_CHECKLISTS:
    for name, (config, input_args, landmarks, horizons, _, output) in CHECKLIST_RUNS.items():
        print(f"[run ] checklist {name} -> {output}")
        subprocess.run(checklist_command(config, input_args, landmarks, horizons, output)
                       + ["--overwrite" if CHECKLIST_OVERWRITE else "--resume"],
                       cwd=REPO_ROOT, check=True)

### Checklist results

Statuses distinguish supported models, inadequate event counts, missing optional
comparators, and unsupported horizons. The selected items show whether Gleason or the
trio entered alongside labs. These are horizon-specific results, not integrated metrics
on the Cox/XGBoost timeline above, and the two runs use different cohorts and clocks, so
they are not comparable patient for patient.

In [ ]:
for name, (*_, output) in CHECKLIST_RUNS.items():
    metrics_path = output / "metrics.csv"
    if not metrics_path.exists():
        print(f"[skip] checklist {name}: no metrics.csv")
        continue
    print(f"=== {name} ===")
    metrics = pd.read_csv(metrics_path)
    display(metrics[[c for c in ["endpoint", "landmark_days", "population", "horizon_days",
                                 "model", "status", "auc", "brier", "n_items", "reason"] if c in metrics]])
    for checklist in sorted(output.glob("*/landmark_*/full/horizon_*/checklist.txt")):
        print(f"--- {checklist.relative_to(output)}")
        print(checklist.read_text())